# Semi-mechanistic Residual Stream Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielyxyang/semimech/blob/main/notebooks/residual_stream_analysis.ipynb)

This notebook provides a comprehensive suite for extracting and analyzing the internal **activation trajectories** from the residual stream of large language models. It identifies "concept directions" using **Readers** (such as PCA or Linear Probing), which extract linear directions in the activation space and project the trajectories onto them to visualize how internal representations evolve across layers and token positions.

**Notebook Structure**
- **Interactive analysis**: A widget-based interface for quick exploration and instant insights without writing code.
- **Manual analysis**: A flexible API-based approach for more customized analysis, batch processing, and fine-grained control over experimental setups.

## Setup

This notebook is designed to work both locally and in Google Colab. We recommend the usage of a GPU for faster extraction of activations. If you plan to use gated models from Hugging Face, make sure you have your `HF_TOKEN` environment variable set or are logged into Hugging Face.

- **Local usage**: The notebook expects the `semimech` package to be installed in your environment.
- **Colab usage**: The setup cell below will automatically install the necessary dependencies from the GitHub repository.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !uv pip install git+https://github.com/danielyxyang/semimech
else:
    %load_ext autoreload
    %autoreload 2

In [ ]:
import sys
from pathlib import Path

import torch
from ipywidgets import widgets

from semimech.activations import Activations, extract_activations
from semimech.analysis import analyze_per_layer, analyze_per_token
from semimech.data import DATASET_REGISTRY, load_dataset_from_spec
from semimech.model import (
    MODEL_REGISTRY,
    get_token_embeddings,
    get_token_groups,
    load_model_and_tokenizer,
    print_token_groups,
)
from semimech.utils import Cache, select
from semimech.visualizations import plot_per_layer, plot_per_token, plot_reader_statistics, plot_token_embeddings
from semimech.widgets import (
    ActivationsExtractorWidget,
    ActivationsSelectorWidget,
    ReadersSelectorWidget,
    ShowHideCheckbox,
    TokenEmbeddingsLoaderWidget,
    TopKExtractorWidget,
)

torch.set_grad_enabled(False)

if "google.colab" in sys.modules:
    PATH_ACTIVATIONS = Path("activations")
else:
    PATH_ACTIVATIONS = Path("../activations")

## Interactive analysis

The following cell provides an interactive GUI for analyzing the model's residual activations in two steps:

1.  **Extract activations**: Choose a model and dataset to collect the activations. You can further customize this by:
    - Include **custom responses** to see how they affect the activation trajectories.
    - Choose whether to **save/load activations** to/from disk for faster subsequent analysis.
    - Extract **top-k next-token predictions** from various layers for more context.
    - Load **token embeddings** to see where the activations are headed in vocabulary space.

2.  **Analyze activations**: Identify concept directions and project trajectories onto them. You can further customize this by:
    - Choose between analyzing **per-layer** trajectories (across tokens) or **per-token** trajectories (across layers).
    - Select different **reading methods** (e.g., PCA, Linear Probes) to identify concept directions.
    - Choose to analyze and/or visualize **separately** per layer/token or globally across all selected layers/tokens.
    - Use different pooling methods (e.g., `last` for the final token, `all` for the full trajectory) to decide which activations should be considered for analysis and visualization. Use additional options to **exclude BOS** (first token) or **special tokens** (e.g., chat template headers) for cleaner analysis.

> **⚠️ Warning:** Visualizing too many samples at once can significantly slow down the interactive plots. Start with a smaller selection (e.g., 5-10 samples) before expanding.

<details>
<summary><b>Additional details</b></summary>

- **Activation selection**: In the list boxes, use **Shift + Click** to select multiple contiguous items, and **Ctrl/Cmd + Click** to select or unselect separate items. If none are selected, all available samples/layers are used by default.
- **Tokens selection (Pooling method)**: Determines which tokens are evaluated:
  - `mean`: Applies mean pooling across all selected tokens.
  - `indices`: Uses a comma-separated list of chosen indices (e.g., `0, 2, 4`).
  - `slice`: Uses Python slice notation (e.g., `0:10` or `-5::`).
  > **Tip**: It is often helpful to select tokens from the back using negative slicing (e.g. `-X::`). This is particularly relevant for per-token analysis where sequences across all samples must be of the same length to be compared.
- **Same as analysis**: When enabled, the visualization uses the same activations as for the analysis.
- **Plotly interface**: Use the interactive interface provided by Plotly to zoom and pan in the plots. Single-click on legend entries to toggle visibility of specific samples or double-click a legend entry to isolate and hide the rest.
- **Tooltips**: Hovering over points displays rich context including the sample ID, current token and token position, top-k next-token predictions (if extracted), and the full text sequence with the current token highlighted.
</details>

In [ ]:
# @title { display-mode: "form" }

def analyze_interactive():
    CACHE = Cache(maxsize=10, encoders={Activations: lambda x: (sorted(x.samples.index.tolist()), sorted(x.layers))})

    # Define layer 1 widgets for extracting activations
    out_layer1 = widgets.Output()
    w_act_extractor = ActivationsExtractorWidget(
        out=out_layer1,
        search_path=PATH_ACTIVATIONS,
        model="gemma3_1b",
        dataset="xstest",
    )
    w_topk_extractor = TopKExtractorWidget(out=out_layer1, model="gemma3_1b")
    w_emb_loader = TokenEmbeddingsLoaderWidget(out=out_layer1, model="gemma3_1b")
    box_layer1_adv_options = widgets.VBox([w_topk_extractor, w_emb_loader])
    box_layer1 = widgets.VBox(
        [
            widgets.HTML("<big><b>Extract activations</b></big>"),
            w_act_extractor,
            ShowHideCheckbox(box_layer1_adv_options, value=False, description="Show advanced options", indent=False),
            box_layer1_adv_options,
            out_layer1,
        ]
    )

    # Define layer 2 widgets for analyzing activations
    w_analyze_selector = widgets.Dropdown(options=["per layer", "per token"], value="per layer", description="Analyze")
    w_readers_selector = ReadersSelectorWidget()
    w_train_separate = widgets.Checkbox(value=True, description="Analyze separately")
    w_eval_separate = widgets.Checkbox(value=True, description="Visualize separately")

    w_train_selector = ActivationsSelectorWidget()
    w_eval_selector = ActivationsSelectorWidget()
    w_eval_link = widgets.Checkbox(value=True, description="Same as analysis")
    w_share_axes = widgets.Checkbox(value=False, description="Share axes")
    w_ncols = widgets.IntSlider(value=3, min=1, max=10, step=1, description="Columns")
    btn_analyze = widgets.Button(description="Analyze", button_style="primary", disabled=True)
    out_analyze = widgets.Output()
    box_train = widgets.VBox([widgets.HTML("<b>Select activations for analysis</b>"), w_train_selector])
    box_eval = widgets.VBox([widgets.HTML("<b>Select activations for visualization</b>"), w_eval_selector, w_eval_link])
    box_layer2_adv_options = widgets.VBox([w_share_axes, w_ncols])
    box_layer2 = widgets.VBox(
        [
            widgets.HTML("<big><b>Analyze activations</b></big>"),
            w_analyze_selector,
            w_readers_selector,
            w_train_separate,
            w_eval_separate,
            widgets.HBox([box_train, box_eval]),
            ShowHideCheckbox(box_layer2_adv_options, value=False, description="Show advanced options", indent=False),
            box_layer2_adv_options,
            btn_analyze,
            out_analyze,
        ]
    )

    # Register handlers
    def _update_separate(*args):
        if w_analyze_selector.value == "per layer":
            w_train_separate.description = "Analyze separately per layer"
            w_eval_separate.description = "Visualize separately per layer"
        else:
            w_train_separate.description = "Analyze separately per token"
            w_eval_separate.description = "Visualize separately per token"

    w_analyze_selector.observe(_update_separate, names="value")
    _update_separate()

    def _update_activations(*args):
        CACHE.clear()

        # Update widgets
        activations = w_act_extractor.activations
        w_topk_extractor.set_activations(activations)
        w_train_selector.set_activations(activations)
        w_eval_selector.set_activations(activations)
        btn_analyze.disabled = activations is None
        out_analyze.clear_output()

    w_act_extractor.observe(_update_activations, names="value")
    _update_activations()

    def _update_link(*args):
        if w_eval_link.value:
            w_eval_selector.link(w_train_selector)
        else:
            w_eval_selector.unlink()

    w_eval_link.observe(_update_link, names="value")
    _update_link()

    def _do_plot(*args):
        activations = w_act_extractor.activations
        if activations is None:
            return

        with out_analyze:
            out_analyze.clear_output(wait=True)

            if w_analyze_selector.value == "per layer":
                analyze_fn = analyze_per_layer
                plot_fn = plot_per_layer
            else:
                analyze_fn = analyze_per_token
                plot_fn = plot_per_token

            # Train and cache model
            train_activations = activations.select(
                sample_ids=w_train_selector.samples or None,
                layers=w_train_selector.layers or None,
            )
            readers = CACHE(analyze_fn)(
                train_activations,
                w_readers_selector.readers,
                pool_method=w_train_selector.pool_method,
                exclude_bos=w_train_selector.exclude_bos,
                exclude_special_tokens=w_train_selector.exclude_special_tokens,
                separate=w_train_separate.value,
            )

            # Plot explained variance
            xlabel = "Layer" if w_analyze_selector.value == "per layer" else "Token position"
            plot_reader_statistics(readers, xlabel=xlabel)

            # Plot evaluation
            eval_activations = activations.select(
                sample_ids=w_eval_selector.samples or None,
                layers=w_eval_selector.layers or None,
            )
            plot_fn(
                eval_activations,
                readers,
                pool_method=w_eval_selector.pool_method,
                exclude_bos=w_eval_selector.exclude_bos,
                exclude_special_tokens=w_eval_selector.exclude_special_tokens,
                token_embeddings=w_emb_loader.token_embeddings,
                separate=w_eval_separate.value,
                ncols=w_ncols.value,
                share_axes=w_share_axes.value,
            )

    btn_analyze.on_click(_do_plot)

    # Display widgets
    display(widgets.VBox([box_layer1, widgets.HTML("<hr>"), box_layer2]))


analyze_interactive()

## Manual analysis

The manual analysis section provides a code-based workflow using the `semimech` API. This approach offers greater flexibility for customized analyses, batch processing, and fine-grained control over your experimental setups compared to the interactive widgets.

In [ ]:
print("Available models:" + "\n - ".join([""] + list(MODEL_REGISTRY)))
print("Available datasets:" + "\n - ".join([""] + list(DATASET_REGISTRY)))

### Extract activations

In [ ]:
dataset = load_dataset_from_spec("xstest", max_samples=500)

In [ ]:
model, tokenizer = load_model_and_tokenizer("gemma3_1b")

In [ ]:
activations = extract_activations(model, tokenizer, dataset, apply_chat_template=True)
activations.samples

### (optional) Save and load activations

In [ ]:
activations.save(PATH_ACTIVATIONS / "xstest/gemma3_1b/chat")

In [ ]:
activations = Activations.load(PATH_ACTIVATIONS / "xstest/gemma3_1b/chat")
activations.samples

### Analysis of per-layer trajectories

* Example A: Analyze with PCA the per-layer "cross-sections" of per-token trajectories over last token of all samples
* Example B: Analyze with linear probe and PCA the per-layer trajectories over last 6 tokens of 100 safe and 100 unsafe samples

In [ ]:
readers_per_layer = analyze_per_layer(
    ### Example A ###
    activations,
    ["pca", "pca"],
    pool_method="last",
    separate=True,

    # ### Example B ###
    # activations.select(activations.samples_safe.index[:100].tolist() + activations.samples_unsafe.index[:100].tolist()),
    # ["linear_probe", "pca"],
    # pool_method=slice(-6, None),
    # separate=True,
)
plot_reader_statistics(readers_per_layer, xlabel="Layer")

In [ ]:
plot_per_layer(
    ### Example A ###
    activations,
    readers_per_layer,
    pool_method="last",
    separate=True,

    # ### Example B ###
    # activations.select(activations.samples_safe.index[:100].tolist() + activations.samples_unsafe.index[:100].tolist()),  # train samples
    # # activations.select(activations.samples_safe.index[100:103].tolist() + activations.samples_unsafe.index[100:103].tolist()),  # test samples
    # readers_per_layer,
    # pool_method=slice(-6, None),
    # separate=True,

    ### Optional: Visualize token embeddings ###
    token_embeddings=get_token_embeddings(model, tokenizer, select(range(tokenizer.vocab_size), at_most=0.25)),
)

### Analysis of per-token trajectories

* Example A: Analyze with PCA per-token trajectories over all tokens of the first sample.
* Example B: Analyze with PCA per-token trajectories over last 10 tokens of 3 safe and 3 unsafe samples.

In [ ]:
readers_per_token = analyze_per_token(
    ### Example A ###
    activations.select(activations.samples.index[0]),
    ["pca", "pca"],
    pool_method="all",
    separate=False,

    ### Example B ###
    # activations.select(activations.samples_safe.index[:3].tolist() + activations.samples_unsafe.index[:3].tolist()),
    # ["pca", "pca"],
    # pool_method=slice(-10, None),
    # separate=True,
)
plot_reader_statistics(readers_per_token, xlabel="Token position")

In [ ]:
plot_per_token(
    ### Example A ###
    activations.select(activations.samples.index[0]),
    readers_per_token,
    pool_method="all",
    separate=False,

    ### Example B ###
    # activations.select(activations.samples_safe.index[:3].tolist() + activations.samples_unsafe.index[:3].tolist()),
    # readers_per_token,
    # pool_method=slice(-10, None),
    # separate=True,

    ### Optional: Visualize token embeddings ###
    # token_embeddings=get_token_embeddings(model, tokenizer, select(range(tokenizer.vocab_size), at_most=0.25)),
)

### (optional) Analysis of token embeddings

In [ ]:
token_groups = get_token_groups(tokenizer)
print_token_groups(tokenizer, token_groups)

In [ ]:
plot_token_embeddings(
    model,
    tokenizer,
    token_groups,
    num_components=8,
)

### (optional) Generate responses

In [ ]:
from semimech.activations import _prepare_sample


def generate(model, tokenizer, sample, include_response, apply_chat_template):
    sample_prepared = _prepare_sample(
        sample, tokenizer, include_response=include_response, apply_chat_template=apply_chat_template
    )
    inputs = tokenizer(sample_prepared["input"], return_tensors="pt", add_special_tokens=not apply_chat_template)
    inputs = inputs.to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=10)
    response = tokenizer.decode(outputs)[0]
    return response


def test_generate(model, tokenizer, sample, include_responses=[False, "Sure", "Sorry"], apply_chat_template=True):
    print(f"ID:             {sample['id']}")
    print(f"is_safe:        {sample['is_safe']}")
    print(f"is_adversarial: {sample['is_adversarial']}")
    for include_response in include_responses:
        print("-" * 50)
        response = generate(
            model,
            tokenizer,
            sample,
            include_response=include_response,
            apply_chat_template=apply_chat_template,
        )
        print(response)


test_generate(model, tokenizer, dataset[0], apply_chat_template=True)